In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/murtazaabdullah2010/kz-task-1-llm-based-text/kz_test.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/train_data.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/test_data.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/sample_output.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/custom_archive.py


In [2]:
train_ds= pd.read_csv("/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/train_data.csv")
test_ds = pd.read_csv("/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/test_data.csv")
sample_sub = pd.read_csv("/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/sample_output.csv")

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel

In [17]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
class DF(nn.Module):
    def __init__(self, df, tokenizer):
        self.df = df
        self.cpp = self.df["cpp_source"]
        self.py = self.df["py_source"]
        self.tok = tokenizer
    def __len__(self, ):
        return len(self.df)
    def __getitem__(self, idx):
        cpp = self.cpp[idx]
        py = self.py[idx]
        tok1= self.tok(py, max_length = 256, padding = "max_length", truncation = True, return_tensors = "pt")
        tok2 = self.tok(cpp, max_length= 256, padding = "max_length", truncation = True ,return_tensors = "pt")
        input_ids1, attn_mask1 = tok1["input_ids"].squeeze(0), tok1["attention_mask"].squeeze(0)
        input_ids2, attn_mask2 = tok2["input_ids"].squeeze(0), tok2["attention_mask"].squeeze(0)
        return input_ids1, attn_mask1, input_ids2, attn_mask2
train_df = DF(train_ds, tokenizer)
train_dl  = DataLoader(train_df, batch_size= 16, shuffle = True, num_workers = 4, pin_memory = True)
for batch in train_dl:
    x1, x2, y1, y2= batch
    print(x1.shape)
    print(y1.shape)
    break

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

torch.Size([16, 256])
torch.Size([16, 256])


In [18]:
AutoModel.from_pretrained("answerdotai/ModernBERT-base")

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ModernBertModel(
  (embeddings): ModernBertEmbeddings(
    (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (layers): ModuleList(
    (0): ModernBertEncoderLayer(
      (attn_norm): Identity()
      (attn): ModernBertAttention(
        (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
        (Wo): Linear(in_features=768, out_features=768, bias=False)
        (out_drop): Identity()
      )
      (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): ModernBertMLP(
        (Wi): Linear(in_features=768, out_features=2304, bias=False)
        (act): GELUActivation()
        (drop): Dropout(p=0.0, inplace=False)
        (Wo): Linear(in_features=1152, out_features=768, bias=False)
      )
    )
    (1-21): 21 x ModernBertEncoderLayer(
      (attn_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): ModernBertAt

In [19]:
class MODEL(nn.Module):
    def __init__(self):
        super().__init__()
        self.back = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
        self.fc = nn.Linear(768, 256)
        self.fc= nn.Sequential(
            nn.Linear(768, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, 256)
        )
    def forward(self, x1, x2):
        x = self.back(input_ids=x1, attention_mask=x2).last_hidden_state[:, 0, :]
        return self.fc(x)
device = ("cuda" if torch.cuda.is_available() else "cpu")


In [20]:
torch.cuda.empty_cache()

In [37]:
model = MODEL().to(device)
opt =torch.optim.AdamW(model.parameters(), lr= 2e-5)
model.train()
temp = 0.07
for e in range(5):
    t_loss = 0
    for batch in train_dl:
        x1, x2, y1, y2 = batch
        x1, x2, y1, y2 = x1.to(device), x2.to(device), y1.to(device), y2.to(device)
        py = F.normalize(model(x1, x2), dim = 1)
        cpp = F.normalize(model(y1, y2), dim = 1)
        logits = py @ cpp.T
        labels= torch.arange(logits.size(0)).to(device)
        opt.zero_grad()
        loss= F.cross_entropy(logits/temp, labels)
        loss.backward()
        opt.step()
        t_loss  +=loss.item()
    torch.cuda.empty_cache()
    print("E: ", e, "train_loss : ", t_loss/len(train_dl))

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


E:  0 train_loss :  1.9101111718586512
E:  1 train_loss :  1.2546212375164032
E:  2 train_loss :  0.6475567753825869
E:  3 train_loss :  0.20065821281501225
E:  4 train_loss :  0.06983333041093179
E:  5 train_loss :  0.03847050967825843
E:  6 train_loss :  0.022617747208901813
E:  7 train_loss :  0.026954259789947952
E:  8 train_loss :  0.010326684801839292
E:  9 train_loss :  0.007661017674503715


In [38]:
cpp_embeds = []
model.eval()
with torch.no_grad():
    for idx, row in test_ds[200:].iterrows():
        code = row["source"]
        tok = tokenizer(code,max_length=256,padding="max_length", truncation=True,return_tensors="pt")
        input_ids = tok["input_ids"].to(device)
        attn_mask = tok["attention_mask"].to(device)
        emb = model(input_ids, attn_mask)  
        emb = F.normalize(emb, dim=1)
        cpp_embeds.append(emb.squeeze(0).cpu())

In [39]:
cpp_embeds = torch.stack(cpp_embeds) 

In [40]:
py_embeds = []

with torch.no_grad():
    for idx, row in test_ds[:200].iterrows():
        code = row["source"]
        tok = tokenizer(code,max_length=256,padding="max_length", truncation=True,return_tensors="pt")
        input_ids = tok["input_ids"].to(device)
        attn_mask = tok["attention_mask"].to(device)
        emb = model(input_ids, attn_mask)
        emb = F.normalize(emb, dim=1)
        py_embeds.append(emb.squeeze(0).cpu())

In [41]:
py_embeds = torch.stack(py_embeds)
sim_mat = py_embeds @ cpp_embeds.T 

In [42]:
results = []
cpp_ids = test_ds[200:]["datapointID"].values
for i in range(sim_mat.shape[0]):
    scores = sim_mat[i]
    top_indices = torch.topk(scores, k=20).indices.tolist()
    candidates = [cpp_ids[j] for j in top_indices]
    results.append(";".join(map(str, candidates)))

In [43]:
sample_sub["answer"] = results

In [44]:
sample_sub.to_csv("sub7.csv", index= False)